# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 – Freshness multiplier**  
> "The 31–90 day window is the strongest stable freshness band… refreshing mature pages produces 3.2× health and 57× impressions."

*Methodology question:* The label here is `health_score` (a FlyRank composite) and `impressions` (observed). The claim is based on comparing refreshed old pages (365+ days old, updated within 30 days) to unrefreshed ones. However, the comparison is observational, not controlled for topic, intent, or prior popularity. The refreshed group may have been selected because they were already better candidates. Does the design control for selection bias, or is the effect confounded by editorial choice?

*Constructive note:* The paper’s own caveat – "the 361+ bucket is too small and unstable" – is honest. For a stronger claim, a matched cohort (e.g., similar age, impressions, position) or an A/B test on a random subset of mature pages would better isolate the refresh effect.

**Finding 2 – AI‑generated content is not penalised**  
> "Within this mostly AI‑authored portfolio, age‑controlled model cohorts do not show a simple blanket penalty tied only to AI use."

*Methodology question:* The label is `health_score` and `impressions` for different model providers (OpenAI, Gemini). The comparison is age‑controlled, but the provider cohorts are not randomly assigned - they reflect different editorial processes, prompts, and topics. Does the validation design control for topic difficulty or competitive intent? The claim that "process quality matters more than AI vs human" is well‑supported, but the evidence is descriptive, not causal.

*Constructive note:* The paper correctly refrains from making a causal claim. A stronger validation would be a within‑topic, within‑age randomised experiment comparing different generation pipelines. The paper’s honesty about the limitation is commendable.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, I used a grouped split by `client_hash_id` to prevent the same client’s pages from appearing in both train and test. To audit the effect of the split design, I compare two scenarios:
- **Leaky (random) split**: pages are split randomly, allowing client‑specific patterns to leak.
- **Honest (grouped) split**: clients are held out entirely.

The metric is Precision@K (20, 50, 100) on the test set, using the same features (`impressions_prev`, `avg_position_prev`) and the same label (`is_declining_label`). The results show that the leaky split overestimates performance, while the honest split provides a more realistic assessment.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    raise ValueError("HF_TOKEN environment variable not set.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FILTERED = f"(SELECT * FROM {MARCH} WHERE gsc_data_available IS TRUE AND gsc_impressions > 0)"

prev = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_prev,
        SUM(gsc_clicks) AS clicks_prev,
        AVG(gsc_avg_position) AS avg_position_prev
    FROM {FILTERED}
    WHERE report_date < DATE '2026-03-16'
    GROUP BY client_hash_id, content_hash_id
""").df()

curr = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_curr,
        SUM(gsc_clicks) AS clicks_curr
    FROM {FILTERED}
    WHERE report_date >= DATE '2026-03-16'
    GROUP BY client_hash_id, content_hash_id
""").df()

agg = prev.merge(curr, on=["client_hash_id", "content_hash_id"], how="inner")
agg["ctr_prev"] = agg["clicks_prev"] / agg["impressions_prev"].replace(0, np.nan)
agg["ctr_curr"] = agg["clicks_curr"] / agg["impressions_curr"].replace(0, np.nan)
agg["is_declining_label"] = (agg["ctr_curr"] < agg["ctr_prev"]).astype(int)

df = agg.dropna(subset=["ctr_prev", "ctr_curr", "avg_position_prev"]).copy()
df = df[df["impressions_prev"] >= 100].copy()

features = ["impressions_prev", "avg_position_prev"]
X = df[features].fillna(0)
y = df["is_declining_label"].values
groups = df["client_hash_id"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler_r = StandardScaler().fit(X_train_r)
X_train_r_scaled = scaler_r.transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

lr_r = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
lr_r.fit(X_train_r_scaled, y_train_r)
scores_r = lr_r.predict_proba(X_test_r_scaled)[:, 1]

rf_r = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf_r.fit(X_train_r, y_train_r)
scores_rf_r = rf_r.predict_proba(X_test_r)[:, 1]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y[train_idx], y[test_idx]

scaler_g = StandardScaler().fit(X_train_g)
X_train_g_scaled = scaler_g.transform(X_train_g)
X_test_g_scaled = scaler_g.transform(X_test_g)

lr_g = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
lr_g.fit(X_train_g_scaled, y_train_g)
scores_g = lr_g.predict_proba(X_test_g_scaled)[:, 1]

rf_g = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf_g.fit(X_train_g, y_train_g)
scores_rf_g = rf_g.predict_proba(X_test_g)[:, 1]

print("Comparison of Precision@K for Logistic Regression")
print("  K   Leaky (random)   Honest (grouped)")
for k in (20, 50, 100):
    p_r = precision_at_k(scores_r, y_test_r, k)
    p_g = precision_at_k(scores_g, y_test_g, k)
    print(f"{k:3d}  {p_r:.3f}             {p_g:.3f}")

print("\nComparison of Precision@K for Random Forest")
print("  K   Leaky (random)   Honest (grouped)")
for k in (20, 50, 100):
    p_rf_r = precision_at_k(scores_rf_r, y_test_r, k)
    p_rf_g = precision_at_k(scores_rf_g, y_test_g, k)
    print(f"{k:3d}  {p_rf_r:.3f}             {p_rf_g:.3f}")

Comparison of Precision@K for Logistic Regression
  K   Leaky (random)   Honest (grouped)
 20  0.650             0.750
 50  0.640             0.800
100  0.610             0.720

Comparison of Precision@K for Random Forest
  K   Leaky (random)   Honest (grouped)
 20  1.000             0.800
 50  1.000             0.840
100  1.000             0.720


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audit the final feature set (`impressions_prev`, `avg_position_prev`) by injecting two known leaky signals: **CTR in the first window** (`ctr_prev`) and **CTR in the second window** (`ctr_curr`). Together, these two features directly define the label `is_declining_label = (ctr_curr < ctr_prev)`. Including both gives the model perfect information about the label, which should inflate Precision@50 to near 1.000. The honest model without these traps serves as the baseline.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df_leaky = df.copy()
df_leaky["ctr_prev_leak"] = df_leaky["ctr_prev"]
df_leaky["ctr_curr_leak"] = df_leaky["ctr_curr"]

X_train_g = df_leaky.iloc[train_idx][["impressions_prev", "avg_position_prev"]].fillna(0)
X_test_g = df_leaky.iloc[test_idx][["impressions_prev", "avg_position_prev"]].fillna(0)
y_train_g = df_leaky.iloc[train_idx]["is_declining_label"].values
y_test_g = df_leaky.iloc[test_idx]["is_declining_label"].values

X_train_leaky = df_leaky.iloc[train_idx][["impressions_prev", "avg_position_prev", "ctr_prev_leak", "ctr_curr_leak"]].fillna(0)
X_test_leaky = df_leaky.iloc[test_idx][["impressions_prev", "avg_position_prev", "ctr_prev_leak", "ctr_curr_leak"]].fillna(0)

rf_honest = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf_honest.fit(X_train_g, y_train_g)
honest_preds = rf_honest.predict_proba(X_test_g)[:, 1]
honest_p50 = precision_at_k(honest_preds, y_test_g, 50)

rf_leaky = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf_leaky.fit(X_train_leaky, y_train_g)
leaky_preds = rf_leaky.predict_proba(X_test_leaky)[:, 1]
leaky_p50 = precision_at_k(leaky_preds, y_test_g, 50)

print(f"Honest Precision@50 (grouped split): {honest_p50:.3f}")
print(f"Leaky Precision@50 (with ctr_prev and ctr_curr added): {leaky_p50:.3f}")
print("The leaky model dramatically inflates performance, confirming these features are unsafe.")

Honest Precision@50 (grouped split): 0.840
Leaky Precision@50 (with ctr_prev and ctr_curr added): 1.000
The leaky model dramatically inflates performance, confirming these features are unsafe.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original bold claim (from Week 5):**  
> "The model predicts whether a page will decline in CTR."

**Rewritten safe claim:**  
> "In this dataset, the model ranks pages by their observed CTR decline between two 15‑day windows. Pages with higher scores are more likely to have declined, but this is a directional signal, not a guarantee. The model’s output is best used as a decision‑support tool to prioritise editorial review, not as a substitute for human judgment or a causal explanation of why a page is declining."

**Why this is safer:**  
- It acknowledges the signal is *observed* in the dataset.
- It frames the output as *directional* rather than deterministic.
- It explicitly limits the use to *decision‑support* and human review.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.